In [ ]:
import pandas as pd
import numpy as np
import os
from glob import glob
import seaborn as sns
import matplotlib.pyplot as plt
from autopath.analysis_smd import SteeredMDAnalysis
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy.ndimage import gaussian_filter1d


In [ ]:
# from scipy.constants import R  # noqa: WPS347
# from scipy.integrate import cumulative_trapezoid
# RT = R * self.temperature / 1e3

In [ ]:
smd_folder = 'sMD'
sysname = '3ptb'
logs = glob(f"{sysname}/{smd_folder}/sMD_*_*_forward.dat")
trajs = glob(f"{sysname}/{smd_folder}/sMD_*_*_forward.dcd")
smd = SteeredMDAnalysis(logs, 
                        sysname, 
                        bin_width=None,#0.02,                
                        dist_minmax=(0.0, 1.8), #nm                        
                        cluster_paths=False,
                        trajectories=trajs,
                        reference_pdb=f"../equilibration/{sysname}/equilibration/{sysname}_equilibrated.pdb",
                        # pocket_select="protein and resid 218 219 262 263 305 306 49 50 91 92 133 134 175 176 and name CA", # my own selection
                        pocket_select='resid 134 135 136 137 138 139 140 157 158 159 160 161 162 180 181 182 183 211 212 213 214 215 226 227 228 229 and name CA',
                        ligand_select='resname UNK and not name H*',
                        dist_column='r_target(nm)',
                        timestep=0.004)
results, gmm_results = smd.run_analysis()
smd.raw_data.to_csv(f"{sysname}/{smd_folder}/sMD_data_raw.csv")
results.head()

### Standard dcTMD

In [ ]:
# DO THIS BY SPEED
fig, ax = plt.subplots(figsize=(12, 4), ncols=len(results['speed'].unique()), nrows=1, sharey=True, sharex=True)
axes = ax.flatten() if len(results['speed'].unique()) > 1 else [ax]

for i, speed in enumerate(sorted(results['speed'].unique())):
    speed_df = results[results['speed'] == speed].copy()
    sns.lineplot(speed_df, x='r_bin', y='Wmean', label='Wmean', ax=axes[i])
    sns.lineplot(speed_df, x='r_bin', y='dG', label='dG', ax=axes[i])
    sns.lineplot(speed_df, x='r_bin', y='Wdiss', label='Wdiss', ax=axes[i])
    axes[i].set_title(f'Speed: {speed} nm/ps', fontsize=12)
    axes[i].set_xlabel('Distance (nm)')
    axes[i].set_ylabel('dG (kJ/mol)')
    if i == 0:
        axes[i].set_ylabel('dG (kJ/mol)')
    if i != len(axes) - 1:
        axes[i].legend_.remove()

plt.xlabel('Distance (nm)'); plt.ylabel('dG (kJ/mol)')
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(12, 4), ncols=len(results['speed'].unique()), nrows=1, sharey=True, sharex=True)
axes = ax.flatten() if len(results['speed'].unique()) > 1 else [ax]

for i, speed in enumerate(sorted(results['speed'].unique())):
    speed_df = results[results['speed'] == speed].copy()
    sns.lineplot(speed_df, x='r_bin', y='Wmean_gmm', label='Wmean_gmm', ax=axes[i])
    sns.lineplot(speed_df, x='r_bin', y='dG_gmm', label='dG_gmm', ax=axes[i])
    sns.lineplot(speed_df, x='r_bin', y='Wdiss_gmm', label='Wdiss_gmm', ax=axes[i])
    axes[i].set_title(f'Speed: {speed} nm/ps', fontsize=12)
    axes[i].set_xlabel('Distance (nm)')
    axes[i].set_ylabel('dG (kJ/mol)')
    if i == 0:
        axes[i].set_ylabel('dG (kJ/mol)')
    if i != len(axes) - 1:
        axes[i].legend_.remove()

plt.xlabel('Distance (nm)'); plt.ylabel('dG (kJ/mol)')
plt.show()
plt.close()

### Jarzynski

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), ncols=len(results['speed'].unique()), nrows=1, sharey=True, sharex=True)
axes = ax.flatten() if len(results['speed'].unique()) > 1 else [ax]

for i, speed in enumerate(sorted(results['speed'].unique())):
    speed_df = results[results['speed'] == speed].copy()
    sns.lineplot(speed_df, x='r_bin', y='Wmean', label='Wmean', ax=axes[i])
    sns.lineplot(speed_df, x='r_bin', y='dG_Jarzynski', label='dG_Jarzynski', ax=axes[i])
    sns.lineplot(speed_df, x='r_bin', y='Wdiss_Jarzynski', label='Wdiss_Jarzynski', ax=axes[i])
    axes[i].set_title(f'Speed: {speed} nm/ps', fontsize=12)
    axes[i].set_xlabel('Distance (nm)')
    axes[i].set_ylabel('kJ/mol')
    if i == 0:
        axes[i].set_ylabel('kJ/mol')
    if i != len(axes) - 1:
        axes[i].legend_.remove()

plt.xlabel('Distance (nm)'); plt.ylabel('kJ/mol')
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(12, 4), ncols=len(results['speed'].unique()), nrows=1, sharey=True, sharex=True)
axes = ax.flatten() if len(results['speed'].unique()) > 1 else [ax]

for i, speed in enumerate(sorted(results['speed'].unique())):
    speed_df = results[results['speed'] == speed].copy()
    sns.lineplot(speed_df, x='r_bin', y='Wmean_gmm', label='Wmean_gmm', ax=axes[i])
    sns.lineplot(speed_df, x='r_bin', y='dG_Jarzynski_gmm', label='dG_Jarzynski_gmm', ax=axes[i])
    sns.lineplot(speed_df, x='r_bin', y='Wdiss_Jarzynski_gmm', label='Wdiss_Jarzynski_gmm', ax=axes[i])
    axes[i].set_title(f'Speed: {speed} nm/ps', fontsize=12)
    axes[i].set_xlabel('Distance (nm)')
    axes[i].set_ylabel('kJ/mol')
    if i == 0:
        axes[i].set_ylabel('kJ/mol')
    if i != len(axes) - 1:
        axes[i].legend_.remove()

plt.xlabel('Distance (nm)'); plt.ylabel('kJ/mol')
plt.show()
plt.close()

In [ ]:
smd.plot_gmm_per_speed(gmm_results)